Se requiere obtener información de las `películas`, el `País` donde se realizó la grabación y cual fue la `productora` encargada de realizarlo.
La información debe ser a partir del año 2010 en adelante de la fecha de lanzamiento, ordenado de manera ascendente por el `título` de la película.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-23")
v_file_date = dbutils.widgets.get("p_file_date")

## 1. Obtenemos las películas y campos que nos interesan (`título`, `presupuesto`, `ingresos obtenidos` y `tiempo de duración` y `fecha de lanzamiento`)

In [0]:
movies_df = spark.read.table("movie_silver.movies").filter(f"file_date = '{v_file_date}'")
movies_filtered_df = (movies_df
    .filter(movies_df.year_release_date >= 2010)
    .select("movie_id", "title", "budget", "revenue", "duration_time", "release_date")
)
display(movies_filtered_df)


movie_id,title,budget,revenue,duration_time,release_date
31007,Welcome to the Rileys,2000000.0,2600000.0,110,2010-10-29
31867,Repo Men,3.2E7,1.8409891E7,111,2010-03-18
32657,Percy Jackson & the Olympians: The Lightning Thief,9.5E7,2.26497209E8,118,2010-02-01
32823,Get Him to the Greek,4.0E7,9.0029656E7,109,2010-06-04
32856,Valentine's Day,5.2E7,2.16485654E8,125,2010-02-10
33217,Diary of a Wimpy Kid,1.5E7,7.5700498E7,92,2010-03-19
33909,Wall Street: Money Never Sleeps,7.0E7,1.34748021E8,133,2010-09-02
34016,She's Out of My League,2.0E7,4.9779728E7,104,2010-03-11
34335,Nine Dead,2000000.0,2600000.0,98,2010-01-01
34563,Our Family Wedding,1.4E7,2.1409028E7,103,2010-03-12


## 2. Obtener el `nombre del país` de la película

In [0]:
production_countries_df = spark.read.table("movie_silver.productions_countries").filter(f"file_date = '{v_file_date}'")
countries_df = spark.read.table("movie_silver.countries")
display(spark.read.table("movie_silver.productions_countries"))
movies_countries_name_df = (
    production_countries_df.join(countries_df, on="country_id", how="inner")
    .select("movie_id", "country_name", "country_id")
)
display(movies_countries_name_df)

movie_id,country_id,ingestion_date,enviroment,file_date
5,214,2026-09-14T20:37:38.298804Z,developer,2024-12-16
11,214,2026-09-14T20:37:38.298804Z,developer,2024-12-16
12,214,2026-09-14T20:37:38.298804Z,developer,2024-12-16
13,214,2026-09-14T20:37:38.298804Z,developer,2024-12-16
14,214,2026-09-14T20:37:38.298804Z,developer,2024-12-16
16,131,2026-09-14T20:37:38.298804Z,developer,2024-12-16
16,152,2026-09-14T20:37:38.298804Z,developer,2024-12-16
16,159,2026-09-14T20:37:38.298804Z,developer,2024-12-16
16,161,2026-09-14T20:37:38.298804Z,developer,2024-12-16
16,151,2026-09-14T20:37:38.298804Z,developer,2024-12-16


movie_id,country_name,country_id
18808,United States of America,214
18823,United States of America,214
18828,United States of America,214
18840,China,146
18840,Germany,151
18840,Russia,204
18840,United Kingdom,162
18840,United States of America,214
18841,United States of America,214
18869,United States of America,214


## 3. Añadir `nombre del país` al DataFrame

In [0]:
movies_final_df = (
    movies_filtered_df.join(movies_countries_name_df, on="movie_id", how="inner")
    .orderBy("release_date", asc=False)
    .select("movie_id", "title", "budget", "revenue", "duration_time", "release_date", "country_name", "country_id")
)

display(movies_final_df)

movie_id,title,budget,revenue,duration_time,release_date,country_name,country_id
34335,Nine Dead,2000000.0,2600000.0,98,2010-01-01,United States of America,214
38357,Morning Glory,4.0E7,5.878518E7,102,2010-01-12,United States of America,214
44040,Devil,1.0E7,3.3583175E7,80,2010-01-13,United States of America,214
40247,Please Give,3000000.0,3750000.0,90,2010-01-22,United States of America,214
44718,Get Low,7500000.0,9375000.0,103,2010-01-22,United States of America,214
44835,Hesher,7000000.0,382946.0,106,2010-01-22,United States of America,214
46838,Tucker and Dale vs Evil,5000000.0,5476793.0,89,2010-01-22,Canada,142
46838,Tucker and Dale vs Evil,5000000.0,5476793.0,89,2010-01-22,United States of America,214
32657,Percy Jackson & the Olympians: The Lightning Thief,9.5E7,2.26497209E8,118,2010-02-01,Canada,142
32657,Percy Jackson & the Olympians: The Lightning Thief,9.5E7,2.26497209E8,118,2010-02-01,United States of America,214


## 4. Obtener la `productora` de la película

In [0]:
movies_companies_df = spark.read.table("movie_silver.movies_companies").filter(f"file_date = '{v_file_date}'")
production_companies_df = spark.read.table("movie_silver.productions_companies").filter(f"file_date = '{v_file_date}'")

movies_genres_name_df = (
    movies_companies_df.join(production_companies_df, on="company_id", how="inner")
    .select("movie_id", "company_name","company_id")
)
display(movies_genres_name_df)

movie_id,company_name,company_id
118957,Story Bridge Films,18988
41515,Picnic Basket,19045
27022,Junction Entertainment,19097
72190,Latina Pictures,19108
164457,Red Granite Pictures,19177
106646,Red Granite Pictures,19177
100042,Red Granite Pictures,19177
146631,Hawthorn Productions,19181
146631,Hawthorne Productions,19182
157386,Andrew Lauren Productions (ALP),19194


## 5. Añadir `nombre de la compañía` al DataFrame

In [0]:
movies_final_df = (
    movies_final_df.join(movies_genres_name_df, on="movie_id", how="inner")
    .select("title", "budget", "revenue", "duration_time", "release_date", "country_name", "company_name", "country_id", "company_id", "movie_id")
)

display(movies_final_df)

title,budget,revenue,duration_time,release_date,country_name,company_name,country_id,company_id,movie_id
Percy Jackson & the Olympians: The Lightning Thief,9.5E7,2.26497209E8,118,2010-02-01,Canada,TCF Vancouver Productions,142,28431,32657
Percy Jackson & the Olympians: The Lightning Thief,9.5E7,2.26497209E8,118,2010-02-01,United States of America,TCF Vancouver Productions,214,28431,32657
Furry Vengeance,3.5E7,3.9340177E7,92,2010-04-02,United Arab Emirates,Furry Vengeance Productions,128,34447,35169
Furry Vengeance,3.5E7,3.9340177E7,92,2010-04-02,United States of America,Furry Vengeance Productions,214,34447,35169
Super 8,5.0E7,2.60095987E8,112,2011-06-08,United States of America,K/O Camera Toys,214,23300,37686
Knight and Day,1.17E8,2.61930431E8,109,2010-06-15,United States of America,Pink Machine,214,21845,37834
Chain Letter,2000000.0,2600000.0,96,2010-10-01,United States of America,Deon Taylor Enterprises,214,26088,38033
Chain Letter,2000000.0,2600000.0,96,2010-10-01,United States of America,Tiger Tail Entertainment,214,26087,38033
Priest,6.0E7,7.8309131E7,87,2011-05-05,United States of America,TOKYOPOP,214,22641,38321
"Big Mommas: Like Father, Like Son",3.2E7,8.3615414E7,107,2011-02-16,United States of America,The Collective Studios,214,21237,38322


## 6. Añadir campo `fecha de creación`

In [0]:
from pyspark.sql.functions import current_timestamp, lit
movies_final_df = ( movies_final_df
                   .withColumn("created_date", lit(v_file_date))
                   .orderBy(movies_final_df.title.asc())
)


In [0]:
display(movies_final_df)

title,budget,revenue,duration_time,release_date,country_name,company_name,country_id,company_id,movie_id,created_date
127 Hours,1.8E7,3.569292E7,94,2010-11-05,United Kingdom,HandMade Films,162,20076,44115,2024-12-23
127 Hours,1.8E7,3.569292E7,94,2010-11-05,United States of America,HandMade Films,214,20076,44115,2024-12-23
5 Days of War,2.0E7,17479.0,113,2011-04-14,United States of America,Rex Media,214,24266,50601,2024-12-23
5 Days of War,2.0E7,17479.0,113,2011-04-14,United States of America,Georgia International Films,214,24264,50601,2024-12-23
5 Days of War,2.0E7,17479.0,113,2011-04-14,United States of America,Dispictures,214,24262,50601,2024-12-23
A Better Life,1.0E7,1759252.0,98,2011-06-24,United States of America,Lime Orchard Productions,214,28708,55720,2024-12-23
A Better Life,1.0E7,1759252.0,98,2011-06-24,United States of America,McLaughlin Films,214,28707,55720,2024-12-23
A Dangerous Method,1.5E7,2.7462041E7,99,2011-09-30,Switzerland,Lago Film,143,19245,48231,2024-12-23
A Dangerous Method,1.5E7,2.7462041E7,99,2011-09-30,United Kingdom,Lago Film,162,19245,48231,2024-12-23
A Dangerous Method,1.5E7,2.7462041E7,99,2011-09-30,Germany,Lago Film,151,19245,48231,2024-12-23


## 7. Escribir datos en el DataLake en formato `Delta`

In [0]:
merge_delta_lake( movies_final_df, "movie_gold", "results_country_prod_company", "tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id AND tgt.created_date = src.created_date", "created_date" )

In [0]:
%sql
SELECT * FROM movie_gold.results_country_prod_company

title,budget,revenue,duration_time,release_date,country_name,company_name,country_id,company_id,movie_id,created_date
#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,Lowland Pictures,214,75278,301325,2024-12-30
#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,AST Studios,214,75277,301325,2024-12-30
8 Days,2000000.0,2600000.0,90,2014-06-15,United States of America,After Eden Pictures,214,85248,433715,2024-12-30
90 Minutes in Heaven,5000000.0,4842699.0,121,2015-09-11,United States of America,Giving Films,214,68117,343795,2024-12-30
A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,Old Bull Pictures,128,53656,241239,2024-12-30
A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,A24,128,41077,241239,2024-12-30
A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,Old Bull Pictures,214,53656,241239,2024-12-30
A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,A24,214,41077,241239,2024-12-30
A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Da Vinci Media Ventures,214,40107,169917,2024-12-30
A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Free State Pictures,214,40106,169917,2024-12-30
